[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [scikit-learn, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/scikit-learn-deep-dive.html)

# Mixed Column Types


## What you will be able to do

Build one pipeline that takes a table of numbers, text and gaps as it arrives, fills the gaps, scales
the numbers, encodes the text, and fits a model, so that new rows go through the same steps.

## The idea

### The problem

The station register exports its table with the flaws a table typed by people has. Some private
stations' elevations are missing. The terrain was typed by hand, so the same word appears as
`inland` and as `INLAND ` with a space after it. A few owners were left blank. And two of the columns
hold words, which no model in scikit-learn can multiply by a coefficient.

Each column needs its own treatment. The numbers need their gaps filled and their scales evened out,
and the words need turning into numbers. Doing that by hand, column by column, before the split,
leaks what the test rows look like into the training, as the **Preprocessing** notebook explained. Doing
it by hand after the split means writing the same steps twice and keeping them in step, and a new
station arriving next month has to go through them a third time.

### What a ColumnTransformer is

> A **`ColumnTransformer`** applies a different transformer to each group of columns and puts the
> results side by side. It takes a list of `(name, transformer, columns)` entries, and like any
> transformer it learns in `fit` and applies what it learned in `transform`. Columns that no entry
> names are dropped unless **`remainder`** says otherwise.

### Why it works that way

A `ColumnTransformer` is a transformer itself, so it can be a step of a pipeline, and the pipeline's
one `fit` call fits every part of it on the training rows alone. The median that fills a missing
elevation, the mean and spread that scale each number, and the list of terrains the encoder knows
are all learned in that call, and used unchanged on the test rows and on any station that arrives
later.

Each entry can be a pipeline of its own. The numbers go through an imputer and then a scaler, and the
words through an imputer that fills a blank with the most common word, and then a one-hot encoder
that makes a column of 0s and 1s for each word it learned. The output of the whole step is one table
of numbers, and `get_feature_names_out` names each of its columns after the entry and the input that
made it, such as `text__terrain_inland`.

The encoder learns the words exactly as they are typed, which is the surprise in a table like this.
To an encoder, `inland` and `INLAND ` are two terrains, and a station typed `Inland` next month is a
third it has never seen. Tidying the text belongs inside the pipeline too, so that every row, old or
new, is tidied the same way before it is encoded.

### Where this shows up

Almost every real table has columns of more than one kind, so most models trained on one need a
step like this. The **Strings** notebook in the **Pandas, Deep Dive** guide tidied text with the
`.str` accessor, which is what the tidying step here does, and the notebook **A Complete Workflow**
builds its model around a pipeline like this one.

### What this notebook covers

- the raw export, its gaps and its inconsistent text
- a pipeline for the numbers and another for the text, joined by a `ColumnTransformer`
- a tidying step inside the pipeline, with `FunctionTransformer`
- fitting, scoring and cross-validating the whole pipeline
- the names of the prepared columns, and the coefficients they carry
- new stations through the same steps, and the model swapped in one line
- five errors, two of them silent

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read it,
and read the output underneath it. Everything from Setup onward is where you start running things.

```python
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

table = pd.DataFrame({"elevation_m": [10.0, 250.0, 900.0], "terrain": ["coastal", "inland", "mountain"]})
prepare = ColumnTransformer([
    ("numbers", StandardScaler(), ["elevation_m"]),
    ("text", OneHotEncoder(), ["terrain"]),
])
print(prepare.fit_transform(table).round(2))
print(prepare.get_feature_names_out())
```

```
[[-1.    1.    0.    0.  ]
 [-0.36  0.    1.    0.  ]
 [ 1.37  0.    0.    1.  ]]
['numbers__elevation_m' 'text__terrain_coastal' 'text__terrain_inland'
 'text__terrain_mountain']
```

The elevation went to the scaler and came out as standard deviations from its mean. The terrain went
to the encoder and came out as three columns, one for each word, with a 1 where a station has that
terrain. One `fit_transform` did both, and the names say which column came from where.

## Setup

Two libraries for the data, scikit-learn, and the tools a pipeline for a mixed table is made of.

- `numpy` and `pandas` build the stations and the export, and the cell prints scikit-learn's version
- `ColumnTransformer` sends each group of columns to its own transformer
- `SimpleImputer` fills gaps, `StandardScaler` scales the numbers and `OneHotEncoder` turns words into
  columns of 0s and 1s
- `FunctionTransformer` makes a pipeline step from a function, here the one that tidies the terrain
- `make_pipeline` joins the steps, and `LinearRegression` and `RandomForestRegressor` are the models
  at the end
- `train_test_split` and `cross_val_score` hold stations back and score the whole pipeline

`station_network` makes a network of 1,200 weather stations in Norway. The stations are made up, and
a generator started from a fixed seed draws them, so the table is the same on every run.
`raw_export` makes the copy a station register would export, also from a fixed seed. It gives each
private station a 30% chance of losing its elevation, each station an 8% chance of having its terrain
retyped in capitals with a space after it, and each owner a 2% chance of being left blank. `raw` is that export. `NUMBERS` and `TEXT` name its two
kinds of feature column.

In [1]:
import numpy as np
import pandas as pd
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

def station_network(n=1200, seed=2025):
    """A made-up network of weather stations in Norway, one row each, with three things to predict."""
    rng = np.random.default_rng(seed)
    latitude = rng.uniform(58.0, 71.0, n)
    longitude = 5 + 1.5 * (latitude - 58) + rng.uniform(0, 5, n)
    coast_km = np.minimum(rng.exponential(25, n), 250)
    elevation = np.clip(rng.gamma(1.5, 150, n) + 3 * coast_km, 0, 1800)
    terrain = np.where(coast_km < 3, "coastal",
              np.where(elevation > 700, "mountain",
              np.where((coast_km < 30) & (rng.uniform(size=n) < 0.6), "fjord", "inland")))
    owner = rng.choice(["met", "road", "airport", "private"], n, p=[0.4, 0.3, 0.1, 0.2])
    installed = rng.integers(1950, 2025, n)
    shift = pd.Series(terrain).map({"coastal": 0.8, "fjord": 0.3, "inland": 0.0, "mountain": -0.5}).to_numpy()
    jan_mean = (1.5 - 0.55 * (latitude - 58) - 0.006 * elevation - 4.0 * (1 - np.exp(-coast_km / 40))
                + shift + rng.normal(0, 0.9, n))
    may_mean = jan_mean + 8.5 + 0.004 * coast_km + rng.normal(0, 1.2, n)
    june_min = 8.5 - 0.35 * (latitude - 58) - 0.0075 * elevation - 0.02 * coast_km + rng.normal(0, 1.5, n)
    table = pd.DataFrame({
        "station": [f"ST{i:04d}" for i in range(1, n + 1)],
        "latitude": latitude.round(2), "longitude": longitude.round(2),
        "elevation_m": elevation.round(0).astype(int), "coast_km": coast_km.round(1),
        "terrain": terrain, "owner": owner, "installed": installed,
        "jan_mean": jan_mean.round(1),
        "snow_on_may_1": may_mean < 2.5,
        "june_frost": june_min < -0.8,
    })
    return table


network = station_network()
FEATURES = ["latitude", "longitude", "elevation_m", "coast_km", "installed"]


def raw_export(network, seed=7):
    """The network as a station register exports it: some private stations' elevations missing,
    terrain typed by hand in several ways, and a few owners left blank."""
    rng = np.random.default_rng(seed)
    raw = network.copy()
    raw["elevation_m"] = raw["elevation_m"].astype(float)
    hidden = (raw["owner"] == "private").to_numpy() & (rng.uniform(size=len(raw)) < 0.3)
    raw.loc[hidden, "elevation_m"] = np.nan
    retyped = rng.uniform(size=len(raw)) < 0.08
    raw.loc[retyped, "terrain"] = raw.loc[retyped, "terrain"].str.upper() + " "
    blank = rng.uniform(size=len(raw)) < 0.02
    raw.loc[blank, "owner"] = np.nan
    return raw


raw = raw_export(network)
NUMBERS = ["latitude", "longitude", "elevation_m", "coast_km", "installed"]
TEXT = ["terrain", "owner"]

print("scikit-learn", sklearn.__version__, "|", len(raw), "stations in the export")

scikit-learn 1.6.1 | 1200 stations in the export


## Worked examples

### The export as it arrives

In [2]:
raw[NUMBERS + TEXT].head(8)

,latitude,longitude,elevation_m,coast_km,installed,terrain,owner
0,70.93,29.07,358.0,17.0,1987,inland,met
1,62.97,16.58,311.0,11.1,2021,fjord,met
2,68.75,21.61,84.0,10.0,2009,inland,airport
3,68.88,22.50,NaN,60.0,1994,inland,private
4,70.69,26.28,461.0,19.3,1970,inland,met
5,59.00,8.16,177.0,40.4,2021,inland,airport
6,62.13,13.73,446.0,128.8,1974,inland,met
7,69.95,26.41,209.0,17.8,1989,FJORD,airport


Station `ST0004`, a private one, has no elevation, and `ST0008`'s terrain reads `FJORD ` in capitals.
Counting the gaps and the words shows how far each flaw goes:

In [3]:
print(raw[NUMBERS + TEXT].isna().sum())
print(raw["terrain"].value_counts())

latitude        0
longitude       0
elevation_m    68
coast_km        0
installed       0
terrain         0
owner          28
dtype: int64
terrain
inland       542
fjord        399
coastal      128
mountain      43
INLAND        40
FJORD         31
COASTAL       14
MOUNTAIN       3
Name: count, dtype: int64


68 elevations and 28 owners are missing, and the terrain has eight spellings for four terrains,
since 88 stations have one in capitals with a space after it.

### A pipeline for the numbers and another for the text

The numbers get the median of their column wherever a value is missing, and then a scaler. The words
get the most common word of their column wherever one is blank, and then an encoder that ignores a
word it did not learn:

In [4]:
numbers = make_pipeline(SimpleImputer(strategy="median"), StandardScaler())
text = make_pipeline(SimpleImputer(strategy="most_frequent"), OneHotEncoder(handle_unknown="ignore"))

prepare = ColumnTransformer([
    ("numbers", numbers, NUMBERS),
    ("text", text, TEXT),
])

Each entry names its part, says what to do and lists its columns. Nothing has been learned yet, since
nothing has been fitted.

### What the step makes of the raw text

Fitting it on the training stations and reading the names of its output columns shows what the
encoder learned:

In [5]:
X = raw[NUMBERS + TEXT]
y = raw["jan_mean"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=0)

prepare.fit(X_train)
names = prepare.get_feature_names_out()

print(len(names), "columns")
print([name for name in names if name.startswith("text__terrain")])

17 columns
['text__terrain_COASTAL ', 'text__terrain_FJORD ', 'text__terrain_INLAND ', 'text__terrain_MOUNTAIN ', 'text__terrain_coastal', 'text__terrain_fjord', 'text__terrain_inland', 'text__terrain_mountain']


The encoder made a column for every spelling it met, eight for terrain, so a fjord typed in capitals
counts as a terrain of its own, with its own coefficient learned from far fewer stations.

### Tidying the text inside the pipeline

A function that tidies the table becomes a pipeline step with `FunctionTransformer`.
`feature_names_out="one-to-one"` tells it that its output columns keep their input's names, which
`get_feature_names_out` needs:

In [6]:
def tidy_text(table):
    """The terrain in lower case with no spaces around it, so each terrain has one spelling."""
    return table.assign(terrain=table["terrain"].str.strip().str.lower())


model = make_pipeline(
    FunctionTransformer(tidy_text, feature_names_out="one-to-one"),
    ColumnTransformer([("numbers", numbers, NUMBERS), ("text", text, TEXT)]),
    LinearRegression(),
)
model.fit(X_train, y_train)

print("R² on test rows:", round(model.score(X_test, y_test), 3))
print(len(model[:-1].get_feature_names_out()), "columns")

R² on test rows: 0.904
13 columns


The tidied pipeline makes 13 columns: five numbers, four terrains and four owners. `model[:-1]` is the
pipeline without its last step, the part that prepares the data. Every step was fitted on the
training rows, and `score` sent the test rows through the same tidying, filling, scaling and encoding
before predicting.

### Cross-validating the whole pipeline

Because the pipeline is one estimator, `cross_val_score` refits all of it inside every fold, so no
median or scale is ever learned from the fold it is scored on:

In [7]:
scores = cross_val_score(model, X_train, y_train, cv=5)

print(scores.round(3))
print("mean:", round(scores.mean(), 3))

[0.894 0.896 0.853 0.875 0.897]
mean: 0.883


### The prepared columns and their coefficients

In [8]:
coefficients = pd.Series(model[-1].coef_, index=model[:-1].get_feature_names_out())

print(coefficients.round(3))

numbers__latitude        -2.028
numbers__longitude       -0.003
numbers__elevation_m     -1.020
numbers__coast_km        -0.916
numbers__installed        0.071
text__terrain_coastal     1.048
text__terrain_fjord       0.151
text__terrain_inland     -0.234
text__terrain_mountain   -0.964
text__owner_airport      -0.008
text__owner_met           0.086
text__owner_private      -0.031
text__owner_road         -0.048
dtype: float64


Each number's coefficient is now per standard deviation, since the numbers were scaled, so latitude's
-2.028 is the change for one standard deviation further north, and longitude's -0.003 is next to
nothing. The terrain coefficients mean something only against each other, because a station has a 1
in exactly one of the four terrain columns. A coastal station is predicted 2.01 degrees warmer than a
mountain station with the same numbers, 1.048 against -0.964.

### New stations through the same steps

A planned station is typed the way the register types, with its terrain in capitals and no owner yet:

In [9]:
planned = pd.DataFrame({
    "latitude": [69.5], "longitude": [23.0], "elevation_m": [450.0], "coast_km": [21.0],
    "installed": [2026], "terrain": ["MOUNTAIN "], "owner": [np.nan],
})

print(model.predict(planned).round(1))

[-9.4]


The pipeline tidied `MOUNTAIN ` to `mountain`, filled the blank owner with the most common owner it
learned, scaled the numbers with the training means, and predicted. Nothing about the planned station
had to be prepared by hand.

### Another model at the end of the same pipeline

Only the last step changes:

In [10]:
forest = make_pipeline(
    FunctionTransformer(tidy_text, feature_names_out="one-to-one"),
    ColumnTransformer([("numbers", numbers, NUMBERS), ("text", text, TEXT)]),
    RandomForestRegressor(random_state=0),
)
forest.fit(X_train, y_train)

print("R² on test rows:", round(forest.score(X_test, y_test), 3))

R² on test rows: 0.879


In [11]:
for name, candidate in [("linear", model), ("forest", forest)]:
    scores = cross_val_score(candidate, X_train, y_train, cv=5)
    print(name, scores.round(3), "mean", round(scores.mean(), 3))

linear [0.894 0.896 0.853 0.875 0.897] mean 0.883
forest [0.86  0.854 0.835 0.866 0.883] mean 0.86


The forest scored 0.879 on the test rows and the linear model 0.904. One split of 300 stations is too
few to rank two models, and choosing between them is cross-validation's job rather than the test
set's. On five folds of the training stations the linear model scored higher in every fold, with a
mean of 0.883 against the forest's 0.86, so on this table the simpler model is the better choice.
Everything before the last step stayed the same.

## Your turn

Five tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than
getting there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/scikit-learn-deep-dive/12-mixed-column-types-solutions.ipynb).

**1.** Print how many stations with a missing elevation each owner has. Use `raw` and `isna`.

In [12]:
# your code here

**2.** Give the numbers' imputer `add_indicator=True`, fit the whole pipeline on the training rows, and print the names of the prepared columns. One more column marks the stations whose elevation was filled.

In [13]:
# your code here

**3.** Build the same pipeline with `LogisticRegression()` in place of `LinearRegression()` to predict `snow_on_may_1`, and print its accuracy on a held-out quarter of the stations, with `random_state=0` for the split.

In [14]:
# your code here

**4.** Predict the January mean of a station typed with terrain `Fjord`, owner `road`, latitude 61.2, longitude 7.1, elevation 120 m, 3.5 km from the coast, installed in 1999, with `model`.

In [15]:
# your code here

**5.** Build the `ColumnTransformer` with `make_column_selector(dtype_include="number")` and `make_column_selector(dtype_include=object)`, imported from `sklearn.compose`, in place of the two lists of names, and check that `get_feature_names_out` gives the same 13 names.

In [16]:
# your code here

## Common errors

### ValueError: A given column is not a column of the dataframe

In [17]:
misspelled = ColumnTransformer([("numbers", numbers, ["Latitude", "elevation_m"])])
misspelled.fit(X_train)

ValueError: A given column is not a column of the dataframe

`Latitude`, with a capital, is not a column of `X_train`, whose column is `latitude`. The `ValueError`
does not say which name was wrong. The `KeyError: 'Latitude'` that caused it, printed higher up the
traceback, does, so read the traceback from the top or compare each list with `X_train.columns`.

### ValueError: Input X contains NaN.

In [18]:
unfilled = make_pipeline(
    FunctionTransformer(tidy_text, feature_names_out="one-to-one"),
    ColumnTransformer([("numbers", StandardScaler(), NUMBERS), ("text", text, TEXT)]),
    LinearRegression(),
)
unfilled.fit(X_train, y_train)

ValueError: Input X contains NaN.
LinearRegression does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values

The numbers went to a scaler with no imputer before it. `StandardScaler` passes a gap through as
`NaN`, and `LinearRegression` raises at the first one. The rest of the message suggests the fix, an
imputer in the pipeline.

### No error, and the text columns dropped

In [19]:
numbers_only = make_pipeline(
    ColumnTransformer([("numbers", numbers, NUMBERS)]),
    LinearRegression(),
)
numbers_only.fit(X_train, y_train)

print("R² on test rows:", round(numbers_only.score(X_test, y_test), 3))
print(numbers_only[:-1].get_feature_names_out())

R² on test rows: 0.877
['numbers__latitude' 'numbers__longitude' 'numbers__elevation_m'
 'numbers__coast_km' 'numbers__installed']


The `ColumnTransformer` named only the numbers, and its `remainder` is `"drop"` unless told otherwise,
so the terrain and the owner never reached the model, and nothing said so. The score fell from 0.904
to 0.877, a number that gives no sign anything is missing. `get_feature_names_out` is the check,
since it lists every column the model sees.

### ValueError: could not convert string to float: 'inland'

In [20]:
passed_through = make_pipeline(
    FunctionTransformer(tidy_text, feature_names_out="one-to-one"),
    ColumnTransformer([("numbers", numbers, NUMBERS)], remainder="passthrough"),
    LinearRegression(),
)
passed_through.fit(X_train, y_train)

ValueError: could not convert string to float: 'inland'

`remainder="passthrough"` keeps the columns no entry names, as they are, so the words reached
`LinearRegression` unencoded. Give the text an entry of its own with an encoder, as `model` does.

### No error, and the same station typed differently gets different forecasts

In [21]:
untidy = make_pipeline(
    ColumnTransformer([("numbers", numbers, NUMBERS), ("text", text, TEXT)]),
    LinearRegression(),
)
untidy.fit(X_train, y_train)

three_ways = pd.concat([planned.assign(owner="met")] * 3, ignore_index=True)
three_ways["terrain"] = ["inland", "Inland", "INLAND "]

print("without tidying:", untidy.predict(three_ways).round(2))
print("with tidying:   ", model.predict(three_ways).round(2))
print("R² on test rows without tidying:", round(untidy.score(X_test, y_test), 3))

without tidying: [-8.68 -8.46 -9.12]
with tidying:    [-8.7 -8.7 -8.7]
R² on test rows without tidying: 0.903


The same station, typed `inland`, `Inland` and `INLAND `, got three forecasts from the pipeline
without the tidying step. `inland` and `INLAND ` are two terrains it learned, with two coefficients,
and `Inland` is one it never saw, which `handle_unknown="ignore"` turned into no terrain at all. The
tidied pipeline gave all three the same forecast. The untidy pipeline scored 0.903 on the test
rows, against the tidied one's 0.904, which is how a mistake like this survives a score, while each spelling of the
same station, learned or new, gets a forecast of its own.

## Recap

- Send each group of columns to its own pipeline with a `ColumnTransformer`
- Fill gaps with `SimpleImputer`, and encode words with `OneHotEncoder(handle_unknown="ignore")`
- Put a tidying function inside the pipeline with `FunctionTransformer`
- Fit, score and cross-validate the whole pipeline as one estimator
- Read `get_feature_names_out` to see every column the model is given

## What is next

The **Saving a Model** notebook writes a fitted pipeline to a file and loads it back, along with the
version of scikit-learn it needs.

---

&#8592; **Previous:** [Feature Importance](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/scikit-learn-deep-dive/11-feature-importance.ipynb)  &nbsp;·&nbsp;  [scikit-learn, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/scikit-learn-deep-dive.html)  &nbsp;·&nbsp;  **Next:** [Saving a Model](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/scikit-learn-deep-dive/13-saving-a-model.ipynb) &#8594;
